# Download historical data from Binance (USDM Futures)

In [4]:
from tabnanny import verbose

import requests
import datetime
import time
import os
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
print(os.getcwd())

/mnt/c/Users/Alexander/PycharmProjects/levbot/Data


#### Sanity check

In [18]:
baseurl = "https://data.binance.vision/?prefix=data/futures/cm/monthly/klines/"
res = requests.head(baseurl)
print(res)

<Response [200]>


#### Generate filenames 

In [26]:
def coinDirURL(coin: str, timeframe: str) -> str:
    """
    Create director URL for the given coin and timeframe.
    :param coin: "BTC"
    :param timeframe: "1m"
    :return: url
    """
    baseurl = "https://data.binance.vision/data/futures/cm/monthly/klines/"
    return f"{baseurl + coin}USD_PERP/{timeframe}/"

def getAvailable(coin: str, timeframe: str) -> list:
    """
     Check available months, counting down from the current date
    :param coin: BTC
    :param timeframe: 1m
    :return: urls of files that can be downloaded
    """
    directory = coinDirURL(coin, timeframe)
    month = int(datetime.datetime.now().strftime("%m"))
    year = int(datetime.datetime.now().strftime("%Y"))
    
    fileprefix = f"{coin}USD_PERP-{timeframe}"
    
    result = []
    while True: # infinite loop until we break
        time.sleep(0.05)
        month -= 1
        # Loop back to december
        if month == 0:
            year -= 1
            month = 12
        
        # URL of the file we are checking, formatting month to have leading zeros
        fileurl = f"{directory+fileprefix}-{year}-{format(month, '02d')}.zip"
        
        # Send the request
        res = requests.head(fileurl)
        print(f"{year} - {month}, response {res.status_code}")
        
        # Check if it exists
        if res.status_code != 200:
            # Does not exist, no more data, return what we have
            return result
        else:
            # Append to results
            result.append(fileurl)
            
        


### Download the files

In [28]:
def dwnld(url, timeframe):
    # Create filename from url
    coin = url.split("/")[-1][0:3]
    suffix = timeframe + url[-12:]
    filename = coin + "/zipped/" + suffix
    
    try:
        os.makedirs(coin + "/zipped/")
    except FileExistsError:
        # directory already exists
        pass

    # request!
    response = requests.get(url)
    
    if response.status_code != 200:
        print("Error downloading "+"filename")
    
    try:
        with open(filename, mode="wb") as file:
            file.write(response.content)
            file.close()
        print(f"Downloaded file {filename}")
    except Exception as e:
        print(e)

### Unzip the files 

In [29]:
def unzipall(coins):
    import shutil
    for coin in coins:
        for file in tqdm(os.listdir(coin+"/zipped")):
            finaldirectory = coin + "/csv/" +file.split("-")[0]
            print(finaldirectory)
            try:
                os.makedirs(finaldirectory)
            except FileExistsError:
                # directory already exists
                pass
            if file.endswith(".zip"):
                shutil.unpack_archive(coin+"/zipped/" +file, finaldirectory)


def downloadZipsForTimeframe(coin, timeframe):
    try:
        filenames = getAvailable(coin, timeframe)
        timeframes = []
        for filename in filenames:
            timeframes.append(timeframe)
        
        with ThreadPoolExecutor() as executor:
            executor.map(dwnld, filenames, timeframes)
    except Exception as e:
        print(e)


def downloadhistorical(coins, timeframes):
    from concurrent.futures import ThreadPoolExecutor
    
    with ThreadPoolExecutor() as executor:
        for coin in coins:
            # Create an equivalent array of just the coin for the map
            coinplural = []
            for timeframe in timeframes:
                coinplural.append(coin)
            executor.map(downloadZipsForTimeframe, coinplural, timeframes)
    unzipall(coins)


In [1]:
import BinanceDownloader
baseurl = "https://data.binance.vision/data/futures/cm/monthly/klines/BTCUSD_PERP/1d/"
dwnlder = BinanceDownloader.Downloader(baseurl, "test")

ModuleNotFoundError: No module named 'BinanceDownloader'